# Lección sobre Ciclos Termodinámicos
La termodinámica es una rama de la física que describe los equilibrios a nivel macroscópico. Los ciclos termodinámicos son una serie de procesos termodinámicos tales que el estado inicial y final son el mismo. Los motores de calor, las bombas de calor y los refrigeradores son ejemplos de dispositivos que operan en ciclos termodinámicos.

Cada ciclo se puede representar en varios tipos de diagramas, incluyendo diagramas P-V (Presión-Volumen), P-T (Presión-Temperatura) y S-T (Entropía-Temperatura). Estos diagramas nos ayudan a visualizar los procesos que ocurren en cada ciclo.

Además, discutiremos los coeficientes termodinámicos y las eficiencias de cada ciclo, así como las diferencias entre los ciclos ideales y los reales.

### Objetivos de aprendizaje
1. **Describir** los ciclos de Carnot, Otto, Diesel, Brayton y Rankine (directo e inverso) como secuencias de procesos politrópicos, sobre diagramas $P\text{-}v$ y $T\text{-}s$.
2. **Calcular** $w_{net}=\oint P\,dv$, $q_{in}$, $q_{out}$ y $\eta$ por integración numérica y por formas cerradas, verificando su coincidencencia.
3. **Comparar** las eficiencias de todos los ciclos operando como máquinas térmicas entre una misma fuente ($T_h$) y un mismo sumidero ($T_c$).
4. **Analizar** el compromiso eficiencia–trabajo y el efecto de las constantes de cada ciclo ($r$, $\rho$, $r_p$, $n$).


## Tipos de Ciclos Termodinámicos
Existen varios tipos de ciclos termodinámicos que son fundamentales en la ingeniería y la física. Algunos de los más importantes son:
1. **Ciclo de Carnot:** Es un ciclo idealizado compuesto por cuatro etapas reversibles: dos isotérmicas alternando con dos adiabáticas. Este ciclo es muy importante porque establece el límite de eficiencia que puede tener cualquier motor térmico.
2. **Ciclo de Rankine:** Este ciclo describe el funcionamiento de las plantas de potencia de vapor típicas, como las centrales eléctricas de carbón, nucleares o de biomasa.
3. **Ciclo de Brayton:** Este ciclo describe el funcionamiento de las turbinas de gas, que son la base de las plantas de energía de gas natural y los motores de los aviones a reacción.
4. **Ciclo de Otto:** Este ciclo describe el funcionamiento de los motores de combustión interna de encendido por chispa, que son el tipo de motor más comúnmente utilizado en los automóviles.
5. **Ciclo de Diesel:** Este ciclo describe el funcionamiento de los motores de combustión interna de encendido por compresión, que se utilizan en muchos vehículos pesados, como camiones y autobuses.

## Descripción Matemática de los Ciclos Termodinámicos
A continuación, describiremos matemáticamente los ciclos termodinámicos mencionados anteriormente. Para ello, utilizaremos la biblioteca de Python `CoolProp` que proporciona propiedades termodinámicas de sustancias puras y mezclas, e importamos la libreria al documento con el nombre *CP*.

In [1]:
# Instalación solo en Google Colab; en el entorno local del curso es un no-op
import sys
if "google.colab" in sys.modules:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "CoolProp", "ipywidgets"], check=True)


In [2]:
# === Setup: librerías, propiedades del aire y maquinaria de ciclos ==========
import numpy as np
import matplotlib.pyplot as plt
from CoolProp.CoolProp import PropsSI
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown

# NumPy 2.x eliminó np.trapz; NumPy <2 no tiene np.trapezoid. Shim válido en ambos.
trapz = np.trapezoid if hasattr(np, "trapezoid") else np.trapz

# --- Aire como gas ideal (propiedades vía CoolProp a 300 K, 1 atm) ----------
R_air = 287.0                                            # J/(kg·K)
k_air = (PropsSI("Cpmass", "T", 300, "P", 101325, "Air")
         / PropsSI("Cvmass", "T", 300, "P", 101325, "Air"))
cv_air = R_air / (k_air - 1)
cp_air = k_air * cv_air

# --- Identidad visual fija por ciclo (paleta Okabe-Ito, apta para daltonismo)
COLOR = {"Carnot": "#0072B2", "Otto": "#E69F00", "Diesel": "#009E73",
         "Brayton": "#D55E00", "Rankine": "#CC79A7", "Stirling": "#56B4E9",
         "Refrigeración": "#56B4E9"}

# --- Rama politrópica de gas ideal ------------------------------------------
def rama(P1, T1, n, P2=None, T2=None, v2=None, N=200):
    """Trayectoria (v, P, T) del proceso Pv^n=C desde (P1, T1) hasta el
    extremo dado (exactamente uno de P2, T2 o v2). n=np.inf -> isocórico."""
    v1 = R_air * T1 / P1
    if np.isinf(n):                                       # isocórico
        if T2 is None:
            T2 = P2 * v1 / R_air
        T = np.linspace(T1, T2, N)
        v = np.full(N, v1)
        return v, R_air * T / v, T
    if v2 is None:
        if T2 is not None and P2 is None:
            if np.isclose(n, 1.0):
                raise ValueError("en un isotérmico especifique P2 o v2")
            P2 = P1 * (T2 / T1) ** (n / (n - 1))
        if np.isclose(n, 0.0):                            # isobárico
            v2 = R_air * T2 / P2
        else:
            v2 = v1 * (P1 / P2) ** (1 / n)
    v = np.linspace(v1, v2, N)
    P = P1 * (v1 / v) ** n
    return v, P, P * v / R_air

def w_rama(P1, T1, P2, T2, n):
    """Trabajo específico [J/kg] de la rama (formas cerradas de la Lección 1)."""
    if np.isinf(n):
        return 0.0
    if np.isclose(n, 1.0):
        return R_air * T1 * np.log((R_air * T2 / P2) / (R_air * T1 / P1))
    return R_air * (T2 - T1) / (1 - n)

def q_rama(P1, T1, P2, T2, n):
    """Calor específico [J/kg] de la rama: primera ley q = Δu + w."""
    return cv_air * (T2 - T1) + w_rama(P1, T1, P2, T2, n)

def ensamblar(ramas_spec, P0, T0):
    """Construye un ciclo cerrado de gas ideal.
    ramas_spec: lista de (n, kwargs_extremo) que debe regresar al estado inicial.
    Devuelve trayectorias, estados, w/q por rama, w_net, q_in, q_out, eta y las
    integrales numéricas de verificación (∮P dv y ∮T ds)."""
    Ps, Ts = [P0], [T0]
    vs, PP, TT, ns = [], [], [], []
    P, T = P0, T0
    for n, kw in ramas_spec:
        v_a, P_a, T_a = rama(P, T, n, **kw)
        vs.append(v_a); PP.append(P_a); TT.append(T_a); ns.append(n)
        P, T = P_a[-1], T_a[-1]
        Ps.append(P); Ts.append(T)
    v = np.concatenate(vs); P = np.concatenate(PP); T = np.concatenate(TT)
    s = cp_air * np.log(T / T0) - R_air * np.log(P / P0)      # s(estado 1) = 0
    w_r = [w_rama(Ps[i], Ts[i], Ps[i+1], Ts[i+1], ns[i]) for i in range(len(ns))]
    q_r = [q_rama(Ps[i], Ts[i], Ps[i+1], Ts[i+1], ns[i]) for i in range(len(ns))]
    w_net = sum(w_r)
    q_in = sum(q for q in q_r if q > 0)
    q_out = -sum(q for q in q_r if q < 0)
    return dict(v=v, P=P, T=T, s=s, P_est=Ps, T_est=Ts, n_ramas=ns,
                w_ramas=w_r, q_ramas=q_r, w_net=w_net, q_in=q_in, q_out=q_out,
                eta=w_net / q_in if q_in > 0 else np.nan,
                w_int_Pdv=trapz(P, v), q_int_Tds=trapz(T, s))

# --- Los cuatro ciclos de gas como secuencias politrópicas ------------------
def ciclo_carnot(T_h, T_c, r_iso=3.0, P1=2e6, k=None):
    k = k or k_air
    v1 = R_air * T_h / P1
    v4 = v1 * (T_h / T_c) ** (1 / (k - 1))                # cierre por la adiabática 4-1
    return ensamblar([(1.0, dict(v2=r_iso * v1)), (k, dict(T2=T_c)),
                      (1.0, dict(v2=v4)), (k, dict(T2=T_h))], P1, T_h)

def ciclo_otto(T_c, T_h, r, P1=100e3, k=None):
    k = k or k_air
    v1 = R_air * T_c / P1
    return ensamblar([(k, dict(v2=v1 / r)), (np.inf, dict(T2=T_h)),
                      (k, dict(v2=v1)), (np.inf, dict(T2=T_c))], P1, T_c)

def ciclo_diesel(T_c, T_h, r, P1=100e3, k=None):
    k = k or k_air
    v1 = R_air * T_c / P1                                  # rho queda fijado por T3=T_h
    return ensamblar([(k, dict(v2=v1 / r)), (0.0, dict(T2=T_h)),
                      (k, dict(v2=v1)), (np.inf, dict(T2=T_c))], P1, T_c)

def ciclo_brayton(T_c, T_h, r_p, P1=100e3, k=None):
    k = k or k_air
    return ensamblar([(k, dict(P2=r_p * P1)), (0.0, dict(T2=T_h)),
                      (k, dict(P2=P1)), (0.0, dict(T2=T_c))], P1, T_c)

NOMBRE_PROC = {0.0: "isobárico", 1.0: "isotérmico", np.inf: "isocórico"}

def plot_ciclo(c, nombre, k_usado=None):
    """Diagrama P-v y T-s del ciclo con áreas sombreadas y verificación de
    las integrales ∮P dv y ∮T ds contra las formas cerradas por rama."""
    col = COLOR.get(nombre, "#333333")
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.4))
    ax1.plot(c["v"], c["P"] / 1e3, color=col, lw=2)
    ax1.fill(c["v"], c["P"] / 1e3, color=col, alpha=0.15)
    ax2.plot(c["s"] / 1e3, c["T"], color=col, lw=2)
    ax2.fill(c["s"] / 1e3, c["T"], color=col, alpha=0.15)
    # estados numerados
    T0, P0 = c["T_est"][0], c["P_est"][0]
    for j, (Pj, Tj) in enumerate(zip(c["P_est"][:-1], c["T_est"][:-1]), start=1):
        vj = R_air * Tj / Pj
        sj = cp_air * np.log(Tj / T0) - R_air * np.log(Pj / P0)
        ax1.scatter([vj], [Pj / 1e3], color="black", zorder=5, s=18)
        ax1.annotate(str(j), (vj, Pj / 1e3), textcoords="offset points",
                     xytext=(6, 4), fontsize=9)
        ax2.scatter([sj / 1e3], [Tj], color="black", zorder=5, s=18)
        ax2.annotate(str(j), (sj / 1e3, Tj), textcoords="offset points",
                     xytext=(6, 4), fontsize=9)
    ax1.set_xlabel("$v$ [m³/kg]"); ax1.set_ylabel("$P$ [kPa]")
    ax1.set_title("$P$-$v$ · área encerrada = $w_{net}$"); ax1.grid(alpha=0.3)
    ax2.set_xlabel("$s-s_1$ [kJ/(kg·K)]"); ax2.set_ylabel("$T$ [K]")
    ax2.set_title("$T$-$s$ · área encerrada = $q_{net}=w_{net}$"); ax2.grid(alpha=0.3)
    err_P = 100 * abs(c["w_int_Pdv"] - c["w_net"]) / abs(c["w_net"])
    err_T = 100 * abs(c["q_int_Tds"] - c["w_net"]) / abs(c["w_net"])
    caja = (f"$\\eta$ = {c['eta']*100:.1f} %\n"
            f"$w_{{net}}$ = {c['w_net']/1e3:.1f} kJ/kg\n"
            f"$q_{{in}}$ = {c['q_in']/1e3:.1f} · $q_{{out}}$ = {c['q_out']/1e3:.1f} kJ/kg\n"
            f"$\\oint P\\,dv$ = {c['w_int_Pdv']/1e3:.1f} (err {err_P:.2f} %)\n"
            f"$\\oint T\\,ds$ = {c['q_int_Tds']/1e3:.1f} (err {err_T:.2f} %)")
    ax2.text(0.03, 0.97, caja, transform=ax2.transAxes, va="top", fontsize=9,
             bbox=dict(boxstyle="round", fc="white", ec=col, alpha=0.85))
    fig.suptitle(f"Ciclo {nombre}", fontsize=12)
    plt.tight_layout(); plt.show()
    # desglose de las integrales por rama (demostración numérica)
    print(f"{'rama':^6}{'proceso':^14}{'n':^8}{'w [kJ/kg]':^12}{'q [kJ/kg]':^12}")
    for j, (n, w, q) in enumerate(zip(c["n_ramas"], c["w_ramas"], c["q_ramas"]), 1):
        nom = NOMBRE_PROC.get(n if np.isinf(n) else round(float(n), 3),
                              f"politrópico" if not np.isinf(n) else "isocórico")
        if not np.isinf(n) and abs(n - k_air) < 0.02 or (k_usado and not np.isinf(n)
                                                         and abs(n - k_usado) < 0.02):
            nom = "adiabático"
        print(f"{j}→{j % len(c['n_ramas']) + 1:^4}{nom:^14}{('∞' if np.isinf(n) else f'{n:.2f}'):^8}"
              f"{w/1e3:^12.2f}{q/1e3:^12.2f}")

def plot_dome(fluid, ax, diag="Ts"):
    """Domo de saturación en 'Ts', 'Pv' o 'Ph' (CoolProp)."""
    Tt = PropsSI("Ttriple", fluid) + 0.5
    Tc_ = PropsSI("Tcrit", fluid) - 0.2
    Ts = np.linspace(Tt, Tc_, 200)
    props = {"liq": {}, "vap": {}}
    for key, Q in (("liq", 0), ("vap", 1)):
        props[key]["v"] = 1 / np.array([PropsSI("D", "T", T, "Q", Q, fluid) for T in Ts])
        props[key]["s"] = np.array([PropsSI("S", "T", T, "Q", Q, fluid) for T in Ts]) / 1e3
        props[key]["h"] = np.array([PropsSI("H", "T", T, "Q", Q, fluid) for T in Ts]) / 1e3
    Psat = np.array([PropsSI("P", "T", T, "Q", 0, fluid) for T in Ts])
    if diag == "Ts":
        ax.plot(props["liq"]["s"], Ts - 273.15, color="#777777", lw=1.3)
        ax.plot(props["vap"]["s"], Ts - 273.15, color="#777777", lw=1.3)
        ax.set_xlabel("$s$ [kJ/(kg·K)]"); ax.set_ylabel("$T$ [°C]")
    elif diag == "Pv":
        ax.plot(props["liq"]["v"], Psat / 1e3, color="#777777", lw=1.3)
        ax.plot(props["vap"]["v"], Psat / 1e3, color="#777777", lw=1.3)
        ax.set_xscale("log"); ax.set_yscale("log")
        ax.set_xlabel("$v$ [m³/kg]"); ax.set_ylabel("$P$ [kPa]")
    else:  # Ph
        ax.plot(props["liq"]["h"], Psat / 1e3, color="#777777", lw=1.3)
        ax.plot(props["vap"]["h"], Psat / 1e3, color="#777777", lw=1.3)
        ax.set_yscale("log")
        ax.set_xlabel("$h$ [kJ/kg]"); ax.set_ylabel("$P$ [kPa]")

print(f"Setup OK — aire: R = {R_air:.0f} J/(kg·K), k = {k_air:.4f}, "
      f"cp = {cp_air:.1f}, cv = {cv_air:.1f} J/(kg·K)")


Setup OK — aire: R = 287 J/(kg·K), k = 1.4017, cp = 1001.5, cv = 714.5 J/(kg·K)


## Eficiencia de los Ciclos Termodinámicos
La eficiencia de un ciclo termodinámico se define como la relación entre la energía útil obtenida del ciclo (trabajo) y la energía que se ha tenido que aportar (calor). En términos matemáticos, se puede expresar de la siguiente manera:

$$\eta = \frac{W_{net}}{Q_{in}} = 1 - \frac{Q_{out}}{Q_{in}}$$

donde:
- $\eta$ es la eficiencia del ciclo,
- $W_{net}$ es el trabajo obtenido del ciclo, y
- $Q_{in}$ es el calor aportado al ciclo ($Q_{out}$ es el rechazado al sumidero).

Es importante notar que la eficiencia siempre es un número entre 0 y 1, o expresado en porcentaje, entre 0% y 100%. Un ciclo con una eficiencia del 100% sería un ciclo que convierte toda la energía aportada en trabajo, sin pérdidas. Sin embargo, en la práctica, esto no es posible debido a las pérdidas inherentes en cualquier proceso real, como la fricción y la disipación de calor.


### El escenario comparable: todas las máquinas entre la misma fuente y el mismo sumidero
Para comparar los ciclos **como máquinas térmicas equivalentes**, en toda esta lección los formulamos operando entre una **fuente de calor a $T_h$** y un **sumidero a $T_c$**, imponiendo

$$T_{max}\text{ del ciclo} = T_h, \qquad T_{min}\text{ del ciclo} = T_c.$$

Con esa restricción, cada ciclo queda descrito por $(T_h, T_c)$ más **sus constantes propias** (relación de compresión $r$, relación de corte $\rho$, relación de presiones $r_p$, presión de caldera). La Segunda Ley fija el techo común:

$$\eta \le \eta_{Carnot} = 1 - \frac{T_c}{T_h}.$$

Veremos que cada ciclo **puede acercarse** a ese techo ajustando su constante… pero solo al precio de que su trabajo neto por ciclo tienda a cero.


## Generalización: todo ciclo es una secuencia de procesos politrópicos
Cada rama de los ciclos vistos es un proceso politrópico $Pv^{\,n}=C$ con un valor concreto de $n$ (Lección 1). La tabla identifica el exponente de cada rama y las expresiones de $w$ y $q$ por unidad de masa (gas ideal):

| Ciclo | 1→2 | 2→3 | 3→4 | 4→1 |
|---|---|---|---|---|
| Carnot | $n=1$ ($T_h$) | $n=\gamma$ | $n=1$ ($T_c$) | $n=\gamma$ |
| Stirling | $n=1$ ($T_h$) | $n\to\infty$ (regen.) | $n=1$ ($T_c$) | $n\to\infty$ (regen.) |
| Otto | $n=\gamma$ | $n\to\infty$ | $n=\gamma$ | $n\to\infty$ |
| Diesel | $n=\gamma$ | $n=0$ | $n=\gamma$ | $n\to\infty$ |
| Brayton | $n=\gamma$ | $n=0$ | $n=\gamma$ | $n=0$ |
| Rankine* | bomba (líquido) | $n=0$ | $s=$ cte | $n=0$ |

*En Rankine el fluido cambia de fase: los exponentes politrópicos de gas ideal no aplican dentro del domo, por eso se calcula con entalpías (CoolProp).

**Trabajo y calor de cada rama politrópica** (gas ideal):

$$w_{1\to2}=\frac{R\,(T_2-T_1)}{1-n}\;(n\neq1),\qquad w=RT\ln\frac{v_2}{v_1}\;(n=1),\qquad
q_{1\to2}=c_n\,(T_2-T_1),\quad c_n=c_v\,\frac{n-\gamma}{n-1}.$$

El calor específico politrópico $c_n$ explica los signos: para $1<n<\gamma$, $c_n<0$ (¡el gas se calienta mientras cede calor!). Esto permite el experimento de la celda siguiente: **¿qué pasa con la eficiencia si las compresiones y expansiones no son adiabáticas sino politrópicas?**


In [ ]:
# WIDGET — ciclo tipo Brayton generalizado: compresión y expansión politrópicas
# n = k_air -> adiabático (Brayton clásico); n < k -> compresión refrigerada /
# expansión con aporte de calor. T_h = 1100 K y T_c = 300 K fijos.
def ciclo_generalizado(r_p=8.0, n_comp=1.40, n_exp=1.40):
    T_h, T_c, P1 = 1100.0, 300.0, 100e3
    T2 = T_c * r_p ** ((n_comp - 1) / n_comp)
    T4 = T_h * r_p ** (-(n_exp - 1) / n_exp)
    if T2 >= T_h or T4 <= 0.9 * T_c:
        print(f"Combinación inválida: T2 = {T2:.0f} K, T4 = {T4:.0f} K "
              f"(se requiere T2 < {T_h:.0f} y T4 > {T_c:.0f}). Ajuste r_p o los n.")
        return
    c = ensamblar([(n_comp, dict(P2=r_p * P1)), (0.0, dict(T2=T_h)),
                   (n_exp, dict(P2=P1)), (0.0, dict(T2=T_c))], P1, T_c)
    plot_ciclo(c, "Brayton", k_usado=max(n_comp, n_exp))
    eta_brayton_k = 1 - r_p ** (-(k_air - 1) / k_air)
    print(f"\nη de este ciclo politrópico = {c['eta']:.4f} | "
          f"η Brayton adiabático (mismo r_p) = {eta_brayton_k:.4f} | "
          f"η_Carnot(1100,300) = {1 - 300/1100:.4f}")
    print("Con n_comp < k la compresión se refrigera: cuesta menos trabajo, pero ese "
          "calor rechazado\nno desaparece del balance; observa cómo cambian q_in, q_out "
          "y η respecto al caso adiabático.")

interact(ciclo_generalizado,
         r_p=FloatSlider(8.0, min=2.0, max=20.0, step=0.5, description="r_p"),
         n_comp=FloatSlider(1.40, min=1.05, max=1.55, step=0.01, description="n compresión"),
         n_exp=FloatSlider(1.40, min=1.05, max=1.55, step=0.01, description="n expansión"));


interactive(children=(FloatSlider(value=8.0, description='r_p', max=20.0, min=2.0, step=0.5), FloatSlider(valu…

### Ciclo de Carnot
El ciclo de Carnot consta de cuatro etapas:
1. **Expansión isotérmica**: El gas se expande, realizando trabajo sobre el entorno. La temperatura del gas permanece constante debido al contacto térmico con la fuente de calor de alta temperatura.
2. **Expansión adiabática**: El gas continúa expandiéndose, realizando trabajo sobre el entorno. No hay intercambio de calor con el entorno, por lo que la temperatura del gas disminuye.
3. **Compresión isotérmica**: El gas se comprime, con el trabajo realizado sobre el gas por el entorno. La temperatura del gas permanece constante debido al contacto térmico con la fuente de calor de baja temperatura.
4. **Compresión adiabática**: El gas continúa comprimiéndose, con el trabajo realizado sobre el gas por el entorno. No hay intercambio de calor con el entorno, por lo que la temperatura del gas aumenta.
Al final de este ciclo, el gas ha vuelto a su estado inicial.

El diagrama $P$-$v$ del ciclo de Carnot se genera con código en la celda siguiente (dos isotermas y dos adiabáticas).

La eficiencia del ciclo de Carnot (η) se calcula como:

$$\eta_{Carnot} = 1 - \frac{T_c}{T_h}$$

donde:

- $T_c$ es la temperatura de la fuente fría (en kelvin)
- $T_h$ es la temperatura de la fuente caliente (en kelvin)


In [ ]:
# WIDGET — ciclo de Carnot: P-v y T-s con verificación de integrales
def carnot_interactivo(T_h=800.0, T_c=300.0, r_iso=3.0):
    c = ciclo_carnot(T_h, T_c, r_iso)
    plot_ciclo(c, "Carnot")
    print(f"\nη teórica = 1 - T_c/T_h = {1 - T_c/T_h:.4f}  "
          f"| η del ciclo ensamblado = {c['eta']:.4f}")

interact(carnot_interactivo,
         T_h=FloatSlider(800, min=500, max=1200, step=10, description="T_h [K]"),
         T_c=FloatSlider(300, min=250, max=450, step=5, description="T_c [K]"),
         r_iso=FloatSlider(3.0, min=1.5, max=6.0, step=0.1, description="r isotérmica"));


interactive(children=(FloatSlider(value=800.0, description='T_h [K]', max=1200.0, min=500.0, step=10.0), Float…

### Ciclo de Otto
El ciclo de Otto es el ciclo idealizado que se utiliza para modelar los motores de encendido por chispa (como los motores de gasolina). Este ciclo consta de cuatro procesos principales:
1. **Admisión (Proceso 0-1):** Entra la mezcla de aire y combustible al cilindro.
2. **Compresión (Proceso 1-2):** La mezcla de aire y combustible se comprime adiabáticamente, lo que aumenta su temperatura y presión.
3. **Combustión (Proceso 2-3):** La chispa de la bujía provoca la combustión de la mezcla, lo que aumenta aún más la temperatura y la presión. Este proceso se modela como una adición de calor a volumen constante.
4. **Expansión (Proceso 3-4):** Los gases calientes empujan el pistón, realizando trabajo sobre él. Este proceso se modela como una expansión adiabática.
5. **Escape (Proceso 4-0):** Los gases de escape son expulsados del cilindro, y el ciclo se repite.

La eficiencia del ciclo de Otto se puede expresar en términos del ratio de compresión $r = V_1/V_2$ y la relación de calores específicos $\gamma = c_p/c_v$:

$$\eta = 1 - \frac{1}{r^{\gamma-1}}$$

donde:

- $r$ es la relación de compresión (volumen antes de la compresión / volumen después de la compresión)
- $\gamma$ es el índice de adiabaticidad del aire


In [5]:
# WIDGET — ciclo Otto entre T_c y T_h: la relación de compresión r es el grado de libertad
def otto_interactivo(r=8.0, T_c=300.0, T_h=1400.0):
    T2 = T_c * r ** (k_air - 1)
    if T2 >= T_h:
        r_max = (T_h / T_c) ** (1 / (k_air - 1))
        print(f"r = {r:.1f} ≥ r_max = {r_max:.1f}: la compresión ya alcanza T_h "
              f"(T2 = {T2:.0f} K) y no puede añadirse calor. Reduzca r o aumente T_h.")
        return
    c = ciclo_otto(T_c, T_h, r)
    plot_ciclo(c, "Otto")
    print(f"\nη fórmula = 1 - 1/r^(k-1) = {1 - r**(1 - k_air):.4f} "
          f"| η del ciclo = {c['eta']:.4f} | η_Carnot({T_h:.0f},{T_c:.0f}) = {1 - T_c/T_h:.4f}")

interact(otto_interactivo,
         r=FloatSlider(8.0, min=2.0, max=14.0, step=0.5, description="r = V₁/V₂"),
         T_c=FloatSlider(300, min=250, max=400, step=5, description="T_c [K]"),
         T_h=FloatSlider(1400, min=900, max=2200, step=25, description="T_h [K]"));


interactive(children=(FloatSlider(value=8.0, description='r = V₁/V₂', max=14.0, min=2.0, step=0.5), FloatSlide…

#### Reflexión — Otto: ¿$r$ grande es siempre mejor?
Entre $T_c$ y $T_h$ fijos, $r$ solo puede crecer hasta $r_{max}=(T_h/T_c)^{1/(\gamma-1)}$, donde la compresión ya alcanza $T_h$: allí $\eta \to \eta_{Carnot}$ pero $q_{in}\to 0$ y $w_{net}\to 0$ (¡una máquina perfecta que no produce nada!). El **máximo trabajo por ciclo** ocurre en

$$r^{*} = \left(\frac{T_h}{T_c}\right)^{\frac{1}{2(\gamma-1)}} \quad\Longrightarrow\quad
w_{net}^{max} = c_v\left(\sqrt{T_h}-\sqrt{T_c}\right)^2, \qquad
\eta^{*} = 1 - \sqrt{\frac{T_c}{T_h}}.$$

Compruébalo con el widget: sube $r$ hacia $r_{max}$ y observa cómo el área encerrada en $P$-$v$ se estrecha mientras $\eta$ sube.


### Ciclo Diesel

El ciclo Diesel consta de cuatro procesos principales:

1. **Admisión (Proceso 0-1):** Entra aire al cilindro.
2. **Compresión (Proceso 1-2):** El aire se comprime adiabáticamente, lo que aumenta su temperatura y presión.
3. **Combustión (Proceso 2-3):** Se inyecta combustible en el aire caliente, lo que provoca su combustión y aumenta aún más la temperatura y la presión. Este proceso se modela como una adición de calor a presión constante.
4. **Expansión (Proceso 3-4):** Los gases calientes empujan el pistón, realizando trabajo sobre él. Este proceso se modela como una expansión adiabática.
5. **Escape (Proceso 4-0):** Los gases de escape son expulsados del cilindro, y el ciclo se repite.

La eficiencia del ciclo Diesel se puede expresar en términos del ratio de compresión $r = V_1/V_2$, la relación de calores específicos $\gamma = c_p/c_v$ y el ratio de corte $\rho = V_3/V_2$:

$$\eta = 1 - \frac{1}{r^{\gamma-1}} \cdot \frac{\rho^\gamma - 1}{\gamma (\rho - 1)}$$

In [8]:
# WIDGET — ciclo Diesel entre T_c y T_h: r libre, rho queda fijado por T3 = T_h
def diesel_interactivo(r=16.0, T_c=300.0, T_h=1800.0):
    T2 = T_c * r ** (k_air - 1)
    if T2 >= T_h:
        r_max = (T_h / T_c) ** (1 / (k_air - 1))
        print(f"r = {r:.1f} ≥ r_max = {r_max:.1f}: T2 = {T2:.0f} K alcanza T_h. Reduzca r.")
        return
    rho = T_h / T2                                   # relación de corte inducida
    c = ciclo_diesel(T_c, T_h, r)
    plot_ciclo(c, "Diesel")
    eta_f = 1 - (1 / r**(k_air - 1)) * (rho**k_air - 1) / (k_air * (rho - 1))
    print(f"\nρ inducido = T_h/T2 = {rho:.2f} | η fórmula = {eta_f:.4f} "
          f"| η del ciclo = {c['eta']:.4f} | η_Carnot = {1 - T_c/T_h:.4f}")

interact(diesel_interactivo,
         r=FloatSlider(16.0, min=8.0, max=24.0, step=0.5, description="r = V₁/V₂"),
         T_c=FloatSlider(300, min=250, max=400, step=5, description="T_c [K]"),
         T_h=FloatSlider(1800, min=1000, max=2400, step=25, description="T_h [K]"));


interactive(children=(FloatSlider(value=16.0, description='r = V₁/V₂', max=24.0, min=8.0, step=0.5), FloatSlid…

### Ciclo Brayton
El ciclo de Brayton, también conocido como ciclo de Joule, es el ciclo fundamental de las turbinas de gas. Este ciclo consta de cuatro procesos principales:
1. **Admisión y Compresión (Proceso 1-2):** El aire es aspirado y comprimido adiabáticamente en el compresor, lo que aumenta su temperatura y presión.
2. **Combustión (Proceso 2-3):** Se inyecta combustible en el aire caliente, lo que provoca su combustión y aumenta aún más la temperatura y la presión. Este proceso se modela como una adición de calor a presión constante.
3. **Expansión (Proceso 3-4):** Los gases calientes se expanden adiabáticamente en la turbina, realizando trabajo sobre ella y sobre el compresor.
4. **Escape (Proceso 4-1):** Los gases de escape son expulsados, y el ciclo se repite.

La eficiencia del ciclo Brayton se puede expresar en términos del ratio de presión $r_p = P_2/P_1$ y la relación de calores específicos $\gamma = c_p/c_v$:

$$\eta = 1 - \frac{1}{r_p^{\frac{\gamma-1}{\gamma}}}$$

donde:

- $r_p$ es la relación de presiones (presión después de la compresión / presión antes de la compresión)
- $\gamma$ es el índice de adiabaticidad del aire


In [ ]:
# WIDGET — ciclo Brayton entre T_c y T_h: la relación de presiones r_p es el grado de libertad
def brayton_interactivo(r_p=8.0, T_c=300.0, T_h=1100.0):
    T2 = T_c * r_p ** ((k_air - 1) / k_air)
    if T2 >= T_h:
        rp_max = (T_h / T_c) ** (k_air / (k_air - 1))
        print(f"r_p = {r_p:.1f} ≥ r_p,max = {rp_max:.1f}: T2 = {T2:.0f} K alcanza T_h. Reduzca r_p.")
        return
    c = ciclo_brayton(T_c, T_h, r_p)
    plot_ciclo(c, "Brayton")
    print(f"\nη fórmula = 1 - r_p^-((k-1)/k) = {1 - r_p**(-(k_air-1)/k_air):.4f} "
          f"| η del ciclo = {c['eta']:.4f} | η_Carnot = {1 - T_c/T_h:.4f}")

interact(brayton_interactivo,
         r_p=FloatSlider(8.0, min=2.0, max=30.0, step=0.5, description="r_p = P₂/P₁"),
         T_c=FloatSlider(300, min=250, max=400, step=5, description="T_c [K]"),
         T_h=FloatSlider(1100, min=800, max=1600, step=25, description="T_h [K]"));


interactive(children=(FloatSlider(value=8.0, description='r_p = P₂/P₁', max=30.0, min=2.0, step=0.5), FloatSli…

#### Reflexión — Brayton: la relación de presiones de máximo trabajo
Igual que en Otto, con $T_c$ y $T_h$ fijos el trabajo neto del Brayton se anula en los dos extremos ($r_p\to1$ y $r_p\to r_{p,max}$) y es máximo en

$$r_p^{*}=\left(\frac{T_h}{T_c}\right)^{\frac{\gamma}{2(\gamma-1)}} \quad\Longrightarrow\quad \eta^{*}=1-\sqrt{\frac{T_c}{T_h}}.$$

Esta $r_p^{*}$ es exactamente la que usa la función `eta_brayton` de la sección comparativa: no es la $r_p$ "óptima en eficiencia" (esa sería $r_{p,max}$, con trabajo nulo) sino la de **máximo trabajo neto**, el punto de diseño más razonable.


### Ciclo Rankine
El ciclo de Rankine es el ciclo fundamental de las centrales eléctricas de vapor. Este ciclo consta de cuatro procesos principales:
1. **Admisión y Compresión (Proceso 1-2):** El agua es bombeada desde la condensadora a la caldera a presión constante.
2. **Calentamiento y Vaporización (Proceso 2-3):** El agua se calienta y se vaporiza en la caldera a presión constante.
3. **Expansión (Proceso 3-4):** El vapor se expande en la turbina, realizando trabajo sobre ella.
4. **Condensación (Proceso 4-1):** El vapor se condensa en el condensador a presión constante, y el ciclo se repite.

La eficiencia del ciclo Rankine se puede expresar en términos de las entalpías en los puntos clave del ciclo:

$$\eta = \frac{h_3 - h_4}{h_3 - h_2}$$

El diagrama $T$-$s$ del ciclo Rankine, con el domo de saturación del agua calculado con CoolProp, se genera con código en la celda siguiente.

La eficiencia del ciclo de Rankine (η) se calcula como:

$$\eta = \frac{W_{turbina} - W_{bomba}}{Q_{caldera}} = \frac{(h_3-h_4)-(h_2-h_1)}{h_3-h_2}$$

donde:

- $W_{turbina}$ es el trabajo realizado por la turbina y $W_{bomba}$ el consumido por la bomba
- $Q_{caldera}=h_3-h_2$ es el calor aportado en la caldera

In [14]:
# WIDGET — ciclo Rankine con propiedades reales del agua (CoolProp) sobre el domo
def estados_rankine(P_cald_Pa, P_cond_Pa, T3_K):
    """Estados 1-4 del Rankine ideal simple (bomba y turbina isentrópicas)."""
    F = "Water"
    h1 = PropsSI("H", "P", P_cond_Pa, "Q", 0, F)
    v1 = 1 / PropsSI("D", "P", P_cond_Pa, "Q", 0, F)
    h2 = h1 + v1 * (P_cald_Pa - P_cond_Pa)                 # bomba isentrópica (líquido)
    Tsat_b = PropsSI("T", "P", P_cald_Pa, "Q", 1, F)
    if T3_K <= Tsat_b + 0.5:                               # sin sobrecalentamiento
        h3 = PropsSI("H", "P", P_cald_Pa, "Q", 1, F)
        s3 = PropsSI("S", "P", P_cald_Pa, "Q", 1, F)
    else:
        h3 = PropsSI("H", "P", P_cald_Pa, "T", T3_K, F)
        s3 = PropsSI("S", "P", P_cald_Pa, "T", T3_K, F)
    h4 = PropsSI("H", "P", P_cond_Pa, "S", s3, F)
    x4 = PropsSI("Q", "P", P_cond_Pa, "H", h4, F)
    return dict(h1=h1, h2=h2, h3=h3, s3=s3, h4=h4, x4=x4,
                w_t=h3 - h4, w_b=h2 - h1, q_in=h3 - h2,
                eta=((h3 - h4) - (h2 - h1)) / (h3 - h2), Tsat_b=Tsat_b)

def _traza_isobara(P, h_a, h_b, N=80):
    """(s, T, v) a lo largo de la isóbara P entre entalpías h_a y h_b."""
    F = "Water"
    h = np.linspace(h_a, h_b, N)
    s = np.array([PropsSI("S", "P", P, "H", hh, F) for hh in h]) / 1e3
    T = np.array([PropsSI("T", "P", P, "H", hh, F) for hh in h]) - 273.15
    v = 1 / np.array([PropsSI("D", "P", P, "H", hh, F) for hh in h])
    return s, T, v

def rankine_interactivo(P_cald_MPa=8.0, P_cond_kPa=10.0, T3_C=500.0):
    F = "Water"
    Pb, Pc_ = P_cald_MPa * 1e6, P_cond_kPa * 1e3
    st = estados_rankine(Pb, Pc_, T3_C + 273.15)
    col = COLOR["Rankine"]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 4.6))
    plot_dome(F, ax1, "Ts"); plot_dome(F, ax2, "Pv")
    # tramos: 2→3 caldera (isóbara), 3→4 turbina (isentrópica), 4→1 condensador
    s23, T23, v23 = _traza_isobara(Pb, st["h2"], st["h3"])
    s41, T41, v41 = _traza_isobara(Pc_, st["h4"], st["h1"])
    Pexp = np.geomspace(Pb, Pc_, 60)
    T34 = np.array([PropsSI("T", "P", p, "S", st["s3"], F) for p in Pexp]) - 273.15
    v34 = 1 / np.array([PropsSI("D", "P", p, "S", st["s3"], F) for p in Pexp])
    s34 = np.full_like(Pexp, st["s3"] / 1e3)
    # T-s
    ax1.plot(s23, T23, color=col, lw=2)
    ax1.plot(s34, T34, color=col, lw=2)
    ax1.plot(s41, T41, color=col, lw=2)
    # P-v
    ax2.plot(v23, np.full_like(v23, Pb / 1e3), color=col, lw=2)
    ax2.plot(v34, Pexp / 1e3, color=col, lw=2)
    ax2.plot(v41, np.full_like(v41, Pc_ / 1e3), color=col, lw=2)
    for ax, xs, ys, lbls in ((ax1, [s23[0], s23[-1], s41[0], s41[-1]],
                              [T23[0], T23[-1], T41[0], T41[-1]], "2341"),):
        for x, y, t in zip(xs, ys, lbls):
            ax.scatter([x], [y], color="black", s=18, zorder=5)
            ax.annotate(t, (x, y), textcoords="offset points", xytext=(5, 4), fontsize=9)
    eta_C = 1 - (PropsSI("T", "P", Pc_, "Q", 0, F)) / (T3_C + 273.15)
    caja = (f"$\\eta$ = {st['eta']*100:.1f} %  ($\\eta_{{Carnot}}$ = {eta_C*100:.1f} %)\n"
            f"$w_t$ = {st['w_t']/1e3:.0f} · $w_b$ = {st['w_b']/1e3:.1f} kJ/kg\n"
            f"$q_{{in}}$ = {st['q_in']/1e3:.0f} kJ/kg · $x_4$ = {st['x4']:.3f}")
    ax1.text(0.02, 0.97, caja, transform=ax1.transAxes, va="top", fontsize=9,
             bbox=dict(boxstyle="round", fc="white", ec=col, alpha=0.85))
    ax1.set_title("Rankine — $T$-$s$ sobre el domo (agua)")
    ax2.set_title("Rankine — $P$-$v$ sobre el domo")
    ax1.grid(alpha=0.3); ax2.grid(alpha=0.3, which="both")
    plt.tight_layout(); plt.show()
    if st["x4"] < 0.85:
        print(f"⚠ x4 = {st['x4']:.3f} < 0.85: humedad excesiva a la salida de la turbina "
              f"(erosión de álabes). Aumente T3 o reduzca P de caldera.")

interact(rankine_interactivo,
         P_cald_MPa=FloatSlider(8.0, min=2.0, max=20.0, step=0.5, description="P caldera [MPa]"),
         P_cond_kPa=FloatSlider(10.0, min=5.0, max=100.0, step=5.0, description="P cond [kPa]"),
         T3_C=FloatSlider(500.0, min=250.0, max=600.0, step=10.0, description="T₃ [°C]"));


interactive(children=(FloatSlider(value=8.0, description='P caldera [MPa]', max=20.0, min=2.0, step=0.5), Floa…

### Ciclo Rankine inverso: refrigeración por compresión de vapor
Si el ciclo Rankine se recorre **en sentido inverso**, la máquina consume trabajo para bombear calor del foco frío al caliente: es el ciclo de **refrigeración por compresión de vapor** (nevera, aire acondicionado, bomba de calor). Sus cuatro procesos, con refrigerante R134a:

1. **Evaporador (4→1):** el refrigerante absorbe $q_L$ del espacio frío a $T_{evap}$ (isobárico, dentro del domo).
2. **Compresor (1→2):** compresión isentrópica del vapor hasta la presión de condensación.
3. **Condensador (2→3):** rechaza $q_H$ al ambiente a $T_{cond}$ (isobárico).
4. **Válvula de expansión (3→4):** estrangulamiento isoentálpico ($h_4=h_3$), irreversible.

El desempeño no se mide con $\eta$ sino con el **coeficiente de operación**:

$$\mathrm{COP}_{ref} = \frac{q_L}{w_{comp}} = \frac{h_1-h_4}{h_2-h_1}
\;\le\; \mathrm{COP}_{Carnot} = \frac{T_{evap}}{T_{cond}-T_{evap}}.$$


In [ ]:
# WIDGET — refrigeración por compresión de vapor (R134a) sobre el domo
def refrigeracion_interactiva(T_evap_C=-10.0, T_cond_C=40.0):
    F = "R134a"
    Te, Tk = T_evap_C + 273.15, T_cond_C + 273.15
    if Tk - Te < 5:
        print("T_cond debe superar a T_evap en al menos 5 K.")
        return
    Pe = PropsSI("P", "T", Te, "Q", 1, F)
    Pk = PropsSI("P", "T", Tk, "Q", 1, F)
    h1 = PropsSI("H", "P", Pe, "Q", 1, F); s1 = PropsSI("S", "P", Pe, "Q", 1, F)
    h2 = PropsSI("H", "P", Pk, "S", s1, F); T2 = PropsSI("T", "P", Pk, "S", s1, F)
    h3 = PropsSI("H", "P", Pk, "Q", 0, F); s3 = PropsSI("S", "P", Pk, "Q", 0, F)
    h4 = h3                                                 # válvula: isoentálpico
    s4 = PropsSI("S", "P", Pe, "H", h4, F); x4 = PropsSI("Q", "P", Pe, "H", h4, F)
    COP = (h1 - h4) / (h2 - h1)
    COP_C = Te / (Tk - Te)
    col = COLOR["Refrigeración"]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 4.6))
    plot_dome(F, ax1, "Ts"); plot_dome(F, ax2, "Ph")
    # T-s: 1→2 (isentrópica), 2→3 (isóbara), 3→4 (válvula, discontinua), 4→1 (isóbara-isoterma)
    h23 = np.linspace(h2, h3, 60)
    s23 = np.array([PropsSI("S", "P", Pk, "H", hh, F) for hh in h23]) / 1e3
    T23 = np.array([PropsSI("T", "P", Pk, "H", hh, F) for hh in h23]) - 273.15
    ax1.plot([s1/1e3, s1/1e3], [Te-273.15, T2-273.15], color=col, lw=2)
    ax1.plot(s23, T23, color=col, lw=2)
    ax1.plot([s3/1e3, s4/1e3], [Tk-273.15, Te-273.15], color=col, lw=2, ls="--")
    ax1.plot([s4/1e3, s1/1e3], [Te-273.15, Te-273.15], color=col, lw=2)
    # P-h: rectángulo clásico
    ax2.plot([h1/1e3, h2/1e3], [Pe/1e3, Pk/1e3], color=col, lw=2)
    ax2.plot([h2/1e3, h3/1e3], [Pk/1e3, Pk/1e3], color=col, lw=2)
    ax2.plot([h3/1e3, h4/1e3], [Pk/1e3, Pe/1e3], color=col, lw=2, ls="--")
    ax2.plot([h4/1e3, h1/1e3], [Pe/1e3, Pe/1e3], color=col, lw=2)
    for ax, pts in ((ax1, [(s1/1e3, Te-273.15, "1"), (s1/1e3, T2-273.15, "2"),
                           (s3/1e3, Tk-273.15, "3"), (s4/1e3, Te-273.15, "4")]),
                    (ax2, [(h1/1e3, Pe/1e3, "1"), (h2/1e3, Pk/1e3, "2"),
                           (h3/1e3, Pk/1e3, "3"), (h4/1e3, Pe/1e3, "4")])):
        for x, y, t in pts:
            ax.scatter([x], [y], color="black", s=18, zorder=5)
            ax.annotate(t, (x, y), textcoords="offset points", xytext=(5, 4), fontsize=9)
    caja = (f"COP = {COP:.2f}  (Carnot: {COP_C:.2f})\n"
            f"$q_L$ = {(h1-h4)/1e3:.1f} kJ/kg · $w_{{comp}}$ = {(h2-h1)/1e3:.1f} kJ/kg\n"
            f"$P_{{evap}}$ = {Pe/1e3:.0f} kPa · $P_{{cond}}$ = {Pk/1e3:.0f} kPa · $x_4$ = {x4:.2f}")
    ax1.text(0.02, 0.97, caja, transform=ax1.transAxes, va="top", fontsize=9,
             bbox=dict(boxstyle="round", fc="white", ec=col, alpha=0.85))
    ax1.set_title("Refrigeración (R134a) — $T$-$s$")
    ax2.set_title("Refrigeración (R134a) — $P$-$h$")
    ax1.grid(alpha=0.3); ax2.grid(alpha=0.3, which="both")
    plt.tight_layout(); plt.show()

interact(refrigeracion_interactiva,
         T_evap_C=FloatSlider(-10.0, min=-30.0, max=5.0, step=1.0, description="T evap [°C]"),
         T_cond_C=FloatSlider(40.0, min=25.0, max=60.0, step=1.0, description="T cond [°C]"));


interactive(children=(FloatSlider(value=-10.0, description='T evap [°C]', max=5.0, min=-30.0, step=1.0), Float…

## Comparativa general de eficiencias

En esta sección se fijan las condiciones de operación comunes a todos los ciclos.

In [17]:
# Condiciones de operación (puedes ajustar estos valores o usar ipywidgets más adelante)
T_caliente = 800.0 # K (≈527 °C)
T_fria = 300.0 # K (≈27 °C)
# Relación de presiones útil para Brayton, Rankine, etc.
p_caliente = 15e6 # Pa (presión de caldera de ejemplo; nota: Psat(350 °C) ≈ 16.5 MPa)
p_fria = 100e3 # Pa (≈1 atm)

### Formulación unificada: cada ciclo entre la misma fuente $T_h$ y el mismo sumidero $T_c$
Imponiendo $T_{max}=T_h$ y $T_{min}=T_c$, cada ciclo queda con **un grado de libertad** y dos puntos notables: el **límite de Carnot** (eficiencia máxima, trabajo nulo) y el **máximo trabajo neto**:

| Ciclo | Grado de libertad | $\eta$ | Límite Carnot en | Máximo trabajo en | $\eta$ en máx. trabajo |
|---|---|---|---|---|---|
| Carnot | tamaño del ciclo | $1-\dfrac{T_c}{T_h}$ (siempre) | — | — ($w\propto$ tamaño) | $1-\dfrac{T_c}{T_h}$ |
| Stirling ideal (regenerado) | tamaño | $1-\dfrac{T_c}{T_h}$ | — | — | idem |
| Otto | $r$ | $1-r^{-(\gamma-1)}$ | $r_{max}=\left(\tfrac{T_h}{T_c}\right)^{\frac{1}{\gamma-1}}$ | $r^*=\left(\tfrac{T_h}{T_c}\right)^{\frac{1}{2(\gamma-1)}}$ | $1-\sqrt{T_c/T_h}$ |
| Diesel | $r$ (con $\rho=\tfrac{T_h}{T_c\,r^{\gamma-1}}$) | $1-\dfrac{r^{1-\gamma}(\rho^\gamma-1)}{\gamma(\rho-1)}$ | $r\to r_{max}$ ($\rho\to1$) | numérico | $\approx 1-\sqrt{T_c/T_h}$ |
| Brayton | $r_p$ | $1-r_p^{-(\gamma-1)/\gamma}$ | $r_{p,max}=\left(\tfrac{T_h}{T_c}\right)^{\frac{\gamma}{\gamma-1}}$ | $r_p^*=\left(\tfrac{T_h}{T_c}\right)^{\frac{\gamma}{2(\gamma-1)}}$ | $1-\sqrt{T_c/T_h}$ |
| Rankine | $P_{cald}$, $T_3\le T_h$ | por entalpías (CoolProp) | $T_{cond}\to T_c$, $T_3\to T_h$ | según diseño | — |

**Derivación del máximo trabajo (Otto):** con $T_2=T_c r^{\gamma-1}$ y $T_4=T_h r^{-(\gamma-1)}$,
$w_{net}=c_v\left[(T_h-T_2)-(T_4-T_c)\right]$; $\dfrac{dw_{net}}{dr}=0$ da $r^{*}$ tal que $T_2=\sqrt{T_h T_c}$, de donde $w^{max}_{net}=c_v(\sqrt{T_h}-\sqrt{T_c})^2$ y $\eta^*=1-\sqrt{T_c/T_h}$ (mismo resultado en Brayton con $r_p^*$).

En las funciones de la celda siguiente (conservadas del material original): `eta_otto` usa la $r$ **límite** (por eso devuelve exactamente $\eta_{Carnot}$ — con trabajo nulo), mientras que `eta_brayton` usa la $r_p^*$ de **máximo trabajo**. El widget comparador de más abajo muestra ambas caras: eficiencia **y** trabajo neto.


In [21]:
# WIDGET comparador A — todos los ciclos entre la misma fuente T_h y sumidero T_c.
# Cada ciclo de gas se evalúa en su punto de MÁXIMO trabajo neto (diseño razonable);
# Carnot y Stirling marcan el techo de eficiencia (su w depende del tamaño del ciclo).
def comparar_fuente_sumidero(T_h=800.0, T_c=300.0):
    if T_h - T_c < 100:
        print("Aumente la diferencia T_h - T_c (≥ 100 K).")
        return
    k = k_air
    etas, ws = {}, {}
    etas["Carnot"] = 1 - T_c / T_h;   ws["Carnot"] = np.nan
    etas["Stirling"] = 1 - T_c / T_h; ws["Stirling"] = np.nan
    c = ciclo_otto(T_c, T_h, (T_h / T_c) ** (1 / (2 * (k - 1))))
    etas["Otto"], ws["Otto"] = c["eta"], c["w_net"] / 1e3
    r_max = (T_h / T_c) ** (1 / (k - 1))
    mejor = max((ciclo_diesel(T_c, T_h, r) for r in np.linspace(1.5, 0.97 * r_max, 40)),
                key=lambda d: d["w_net"])
    etas["Diesel"], ws["Diesel"] = mejor["eta"], mejor["w_net"] / 1e3
    c = ciclo_brayton(T_c, T_h, (T_h / T_c) ** (k / (2 * (k - 1))))
    etas["Brayton"], ws["Brayton"] = c["eta"], c["w_net"] / 1e3
    # Rankine: condensador saturado a T_c; caldera con Tsat intermedia; vapor vivo a T_h
    Pcond = PropsSI("P", "T", max(T_c, 278.0), "Q", 0, "Water")
    Tsat_b = min(T_c + 0.7 * (T_h - T_c), 620.0)
    Pb = PropsSI("P", "T", Tsat_b, "Q", 0, "Water")
    st = estados_rankine(Pb, Pcond, T_h)
    etas["Rankine"], ws["Rankine"] = st["eta"], (st["w_t"] - st["w_b"]) / 1e3

    nombres = list(etas)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 4.4))
    ax1.bar(nombres, [etas[n] for n in nombres], color=[COLOR[n] for n in nombres], width=0.62)
    ax1.axhline(etas["Carnot"], ls="--", color="#444444", lw=1.2)
    ax1.text(len(nombres) - 0.4, etas["Carnot"] + 0.015, "límite de Carnot",
             ha="right", fontsize=9, color="#444444")
    for i, n in enumerate(nombres):
        ax1.text(i, etas[n] + 0.015, f"{etas[n]*100:.0f}%", ha="center", fontsize=9)
    ax1.set_ylim(0, 1); ax1.set_ylabel("η")
    ax1.set_title(f"Eficiencia entre T_h = {T_h:.0f} K y T_c = {T_c:.0f} K")
    ax1.grid(alpha=0.25, axis="y")
    con_w = [n for n in nombres if np.isfinite(ws[n])]
    ax2.bar(con_w, [ws[n] for n in con_w], color=[COLOR[n] for n in con_w], width=0.62)
    for i, n in enumerate(con_w):
        ax2.text(i, ws[n] * 1.01, f"{ws[n]:.0f}", ha="center", fontsize=9)
    ax2.set_ylabel("$w_{net}$ [kJ/kg de fluido de trabajo]")
    ax2.set_title("Trabajo neto específico en el diseño de máximo trabajo")
    ax2.grid(alpha=0.25, axis="y")
    plt.tight_layout(); plt.show()
    print("Carnot y Stirling no aparecen en el panel derecho: su trabajo por ciclo depende\n"
          "del 'tamaño' (relación de volúmenes) elegido, no solo de T_h y T_c.\n"
          "Nota: Rankine usa agua (kJ/kg de agua); los demás, aire.")

interact(comparar_fuente_sumidero,
         T_h=FloatSlider(800, min=600, max=1600, step=20, description="T_h [K]"),
         T_c=FloatSlider(300, min=280, max=400, step=5, description="T_c [K]"));


interactive(children=(FloatSlider(value=800.0, description='T_h [K]', max=1600.0, min=600.0, step=20.0), Float…

In [22]:
# WIDGET comparador B — misma fuente/sumidero, jugando con la constante de cada ciclo.
# Aquí las r NO están en su punto de máximo trabajo: muévelas y observa cómo cada
# ciclo se acerca al techo de Carnot solo a costa de su trabajo neto.
def comparar_constantes(T_h=1400.0, T_c=300.0, r_otto=8.0, r_diesel=16.0, rp_brayton=8.0):
    k = k_air
    eta_C = 1 - T_c / T_h
    filas = []
    for nombre, ctor, cte in (("Otto", ciclo_otto, r_otto),
                              ("Diesel", ciclo_diesel, r_diesel),
                              ("Brayton", ciclo_brayton, rp_brayton)):
        T2 = T_c * (cte ** (k - 1) if nombre != "Brayton" else cte ** ((k - 1) / k))
        if T2 >= T_h:
            filas.append((nombre, np.nan, np.nan, f"inválido: T2={T2:.0f} K ≥ T_h"))
            continue
        c = ctor(T_c, T_h, cte)
        filas.append((nombre, c["eta"], c["w_net"] / 1e3, ""))
    nombres = [f[0] for f in filas]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 4.2))
    ax1.bar(nombres, [f[1] for f in filas], color=[COLOR[n] for n in nombres], width=0.55)
    ax1.axhline(eta_C, ls="--", color="#444444", lw=1.2)
    ax1.text(2.4, eta_C + 0.015, f"Carnot = {eta_C*100:.0f}%", ha="right",
             fontsize=9, color="#444444")
    for i, f in enumerate(filas):
        if np.isfinite(f[1]):
            ax1.text(i, f[1] + 0.015, f"{f[1]*100:.1f}%", ha="center", fontsize=9)
    ax1.set_ylim(0, 1); ax1.set_ylabel("η"); ax1.grid(alpha=0.25, axis="y")
    ax1.set_title(f"η con r_Otto={r_otto:.1f}, r_Diesel={r_diesel:.1f}, r_p={rp_brayton:.1f}")
    ax2.bar(nombres, [f[2] for f in filas], color=[COLOR[n] for n in nombres], width=0.55)
    for i, f in enumerate(filas):
        if np.isfinite(f[2]):
            ax2.text(i, f[2] * 1.01, f"{f[2]:.0f}", ha="center", fontsize=9)
    ax2.set_ylabel("$w_{net}$ [kJ/kg]"); ax2.grid(alpha=0.25, axis="y")
    ax2.set_title("Trabajo neto con esas constantes")
    plt.tight_layout(); plt.show()
    for f in filas:
        if f[3]:
            print(f"⚠ {f[0]}: {f[3]} — la compresión sola ya alcanza la fuente; reduzca la constante.")

interact(comparar_constantes,
         T_h=FloatSlider(1400, min=700, max=1800, step=25, description="T_h [K]"),
         T_c=FloatSlider(300, min=260, max=400, step=5, description="T_c [K]"),
         r_otto=FloatSlider(8.0, min=2.0, max=16.0, step=0.5, description="r Otto"),
         r_diesel=FloatSlider(16.0, min=8.0, max=26.0, step=0.5, description="r Diesel"),
         rp_brayton=FloatSlider(8.0, min=2.0, max=40.0, step=1.0, description="r_p Brayton"));


interactive(children=(FloatSlider(value=1400.0, description='T_h [K]', max=1800.0, min=700.0, step=25.0), Floa…

### Reflexión final: minimizar o maximizar el trabajo, ciclo a ciclo
- **Compresión** (Lección 1): entre dos presiones dadas, el trabajo de compresión es **mínimo** si es isotérmica ($n=1$) y **máximo** si es adiabática ($n=\gamma$). Por eso refrigerar el compresor (o interenfriar) beneficia a Brayton y a los reciprocantes.
- **Expansión**: al revés — extraer trabajo conviene con la expansión más "caliente" posible (recalentamiento entre etapas de turbina).
- **En el ciclo completo**, sin embargo, cada mejora local altera el balance $q_{in}/q_{out}$: el widget anterior muestra que refrigerar la compresión ($n_{comp}<\gamma$) reduce el trabajo de compresión pero también cambia el calor a aportar; la eficiencia del ciclo puede subir o bajar según dónde termine ese calor. **No hay atajo hacia Carnot**: todos los ciclos alcanzan $\eta_{Carnot}$ solo en el límite de trabajo nulo, y el diseño real es siempre un compromiso $\eta$–$w_{net}$ (los puntos $r^*$, $r_p^*$ de máximo trabajo, con $\eta^*=1-\sqrt{T_c/T_h}$).


---
## Referencias
- Çengel, Y. & Boles, M. (2015). *Termodinámica*, 8ª ed. McGraw-Hill. (Ejemplos 9-2, 9-5 y 10-1 usados en la verificación.)
- Borgnakke, C. & Sonntag, R. (2013). *Fundamentals of Thermodynamics*, 8ª ed. Wiley.
- Bell, I. et al. (2014). *CoolProp: pure and pseudo-pure fluid thermophysical property library*. Ind. Eng. Chem. Res. 53(6).